创建一个人工数据集，并存储在csv（逗号分隔值）文件,os(opreating system)提供了python与操作系统交互的功能，例如：操作文件夹、获取环境变量、执行系统命令等

In [1]:
import os

os.makedirs(os.path.join('..','data'),exist_ok=True)
#makedirs创建文件夹，os.path.join表示拼接路径：..表示上一级文件夹，然后创作一个data，如果没有exist_ok=True，若文件夹已经存在会报错
data_file=os.path.join('..','data','house_tiny.csv')
#生成csv文件路径，在data下面创建一个house_tiny.csv
with open(data_file,'w') as f:
    f.write('NumRooms,Alley,Price\n')#列名
    f.write('NA,Pave,127500\n')#每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [3]:
!pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 762.0 kB/s eta 0:00:12
   -- ------------------------------------- 0.5/9.7 MB 762.0 kB/s eta 0:00:12
   --- ------------------------------------ 0.8/9.7 MB 633.2 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/9.7 MB 633.2 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/9.7 MB 633.2 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/9.7 MB 633.2 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/9.7 MB 633.2 kB/s eta 0:00:15
   ---- ----------------------------------- 1.0/9.7 MB 465.9 kB/s eta 0:00:19
   ---- ----------------------------------- 1.0/9.7 MB 465.9 kB/s eta 0:00:19
   ---- --------------


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'pandas'

In [6]:
import sys

!{sys.executable} -m pip install pandas

  Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl (9.7 MB)

   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   ---------

In [18]:
import pandas as pd

data=pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


In [19]:
data

,NumRooms,Alley,Price
0,NaN,Pave,127500
1,2.0,NaN,106000
2,4.0,NaN,178100
3,NaN,NaN,140000


为了处理缺失的数据，典型的方法包括插值和删除，这里，我们将考虑插值

In [20]:
inputs,outputs=data.iloc[:,0:2],data.iloc[:,2]#data.iloc是按行列的位置来读取
inputs=inputs.fillna(inputs.mean(numeric_only=True))#fillna补充缺失值NaN，inputs.mean()取平均，numeric...只处理数字部分
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


对于inputs中的类别值或离散值，我们将"NaN"视为一个类别

In [22]:
inputs=pd.get_dummies(inputs,dummy_na=True,dtype=int)#pd.get_dummies将类别数据转换为数字形式，后续说明输出类型，不说明默认为布尔类型
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0           1          0
1       2.0           0          1
2       4.0           0          1
3       3.0           0          1


上述的转换因为神经网络没法直接处理pave这种字符串

现在inputs和outputs中所有的条目都是数值类型，他们可以转换为张量格式

In [23]:
import torch
x,y=torch.tensor(inputs.values),torch.tensor(outputs.values)
x,y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500, 106000, 178100, 140000]))